# Task 2: estancia no exponencial, cierre parcial de camas y balance de predicciones
CC2017 Modelación y Simulación, UVG · Hoja de Trabajo 7 · Grupo 5. **Estado: pendiente.**

Las predicciones del Task 2.1 deben escribirse **antes** de simular y no modificarse después.

| Inciso | Requisito (hoja) |
|---|---|
| 2.1 | (a) Wq con Allen-Cunneen para CS = 0, 1, 2 y en cuál es menos precisa; (b) fórmula del tiempo Td de vaciado de la fila tras el cierre de k_off camas durante Tb días; (c) bloqueo de Erlang B con estancia +25 % y cuántas veces mayor es |
| 2.2 | Estancia determinista, exponencial y lognormal (CS = 2) con la misma media 1/μ; Wq simulado con IC y error relativo de Allen-Cunneen |
| 2.3 | Cierre de camas entre t0 y t0 + Tb (capacidad c − k_off), ≥ 100 réplicas; curva de la cola promedio y tiempo de recuperación (< 1.1 × Lq estacionario); comparación con la fórmula 2.1b |
| 2.4 | Variante sin espera con estancia +25 % (30 réplicas) frente a la predicción 2.1c; tabla de predicciones y hallazgo más sorprendente |

Ya disponible en `src/hospital_uci/`: `teoria.mmc`, `teoria.mmcc`, `teoria.erlang_b`; `simulador.simular(..., servicio=...)` (gancho para la distribución de la estancia, probado solo con valores constantes y la exponencial) y `muestreo_dt` (traza de la cola).
Falta implementar: generadores de estancia determinista y lognormal (validar que su media sea 1/μ), eventos de cierre y reapertura de camas, y el promedio de la cola entre réplicas en función del tiempo.

In [ ]:
import sys, json, pathlib, math
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt

RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from hospital_uci import teoria as T, simulador as S, sir

FIG, DAT = RAIZ / "resultados" / "figuras", RAIZ / "resultados" / "datos"
FIG.mkdir(parents=True, exist_ok=True); DAT.mkdir(parents=True, exist_ok=True)
AZUL, NARANJA, GRIS = "#0072B2", "#D55E00", "#6b7280"      # azul/naranja: distinguibles con daltonismo
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6, "font.size": 10,
                     "axes.titlesize": 11, "legend.frameon": False})
P = T.parametros_grupo(5)
lam, mu, c = P["lam"], P["mu"], P["c"]
RES = {}      # resultados numéricos de este notebook (se guardan en JSON al final)